# Coastal flood step 31: 50-year discounted EAD summary

This notebook applies the **exact same discount-factor method** used in the river flooding analysis to the coastal flooding EAD outputs.

Discounting approach:
- annual EAD values are assumed constant through time
- present value is calculated over `50` years
- discount rate is `10%`
- the discount factor is computed as `sum(1 / (1 + r) ** year for year in range(years + 1))`

This matches the river flooding implementation exactly, including **year 0** in the discount-factor sum.

The notebook works from the current **signed weighted area-distance attribution within 5000 m plus nearest-neighbour fallback beyond the buffer** outputs.
It discounts:
- total coastal EAD summaries
- sector and subsector EAD summaries
- sector, subsector, asset, and mangrove-patch attribution summaries

It does **not** discount return-period event damages, because those are not annualized EAD metrics.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
mangrove_attribution_buffer_m = 5000
discount_years = 50
discount_rate = 0.10
discounted_suffix = "PV_50Y_10pct"

scenario_paths = {
    "minimum": base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates",
    "maximum": base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates",
}

scenario_output_paths = {
    scenario_name: damage_estimates_path / "discounted_50_year_weighted_area_distance"
    for scenario_name, damage_estimates_path in scenario_paths.items()
}
for scenario_output_path in scenario_output_paths.values():
    scenario_output_path.mkdir(parents=True, exist_ok=True)

comparison_output_dir = (
    base_path
    / "dphil_paper_3/results_coastal_scenario_comparison/weighted_area_distance_signed/discounted_50_year_ead"
)
comparison_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def compute_discount_rate(years, discount_rate):
    discount_rates = [1 / (1 + discount_rate) ** year for year in range(years + 1)]
    discount_rates_sum = sum(discount_rates)
    return discount_rates_sum


discount_factor_50_years = compute_discount_rate(discount_years, discount_rate)

pd.DataFrame(
    [
        {
            "Discount_Years": discount_years,
            "Discount_Rate": discount_rate,
            "Discount_Factor": discount_factor_50_years,
        }
    ]
)

In [ ]:
def format_usd_readable(value_usd):
    if pd.isna(value_usd):
        return "NA"
    absolute_value_usd = abs(float(value_usd))
    if absolute_value_usd >= 1e9:
        return f"${value_usd / 1e9:,.2f}B"
    if absolute_value_usd >= 1e6:
        return f"${value_usd / 1e6:,.2f}M"
    if absolute_value_usd >= 1e3:
        return f"${value_usd / 1e3:,.1f}K"
    return f"${value_usd:,.0f}"


def add_discounted_columns(dataframe, annual_columns, discount_factor, discounted_suffix):
    discounted_table = dataframe.copy()
    for annual_column in annual_columns:
        if annual_column not in discounted_table.columns:
            continue
        discounted_table[annual_column] = pd.to_numeric(discounted_table[annual_column], errors="coerce")
        discounted_table[f"{annual_column}_{discounted_suffix}"] = (
            discounted_table[annual_column] * discount_factor
        )
    return discounted_table


def load_scenario_tables(damage_estimates_path, attribution_buffer_m):
    weighted_output_dir = damage_estimates_path / "mangrove_attribution_area_distance_all_sectors_signed"
    return {
        "sector_ead": pd.read_csv(
            damage_estimates_path / "coastal_ead_sector_summary_usd_with_pct_avoided.csv"
        ),
        "subsector_ead": pd.read_csv(
            damage_estimates_path / "coastal_ead_subsector_summary_usd_with_pct_avoided.csv"
        ),
        "asset_ead": pd.read_csv(
            damage_estimates_path / "coastal_ead_asset_level_usd.csv"
        ),
        "sector_attribution": pd.read_csv(
            weighted_output_dir
            / f"attribution_breakdown_by_sector_signed_area_distance_{attribution_buffer_m}m_nn_fallback.csv"
        ),
        "subsector_attribution": pd.read_csv(
            weighted_output_dir
            / f"attribution_breakdown_by_subsector_signed_area_distance_{attribution_buffer_m}m_nn_fallback.csv"
        ),
        "sector_sign": pd.read_csv(
            weighted_output_dir
            / f"sector_attribution_net_sign_profile_signed_area_distance_{attribution_buffer_m}m_nn_fallback.csv"
        ),
        "subsector_sign": pd.read_csv(
            weighted_output_dir
            / f"subsector_attribution_net_sign_profile_signed_area_distance_{attribution_buffer_m}m_nn_fallback.csv"
        ),
        "mangrove_patch": pd.read_csv(
            weighted_output_dir
            / f"mangrove_attribution_total_all_sectors_signed_area_distance_{attribution_buffer_m}m_nn_fallback.csv"
        ),
        "asset_attribution_status": pd.read_csv(
            weighted_output_dir
            / f"asset_attribution_status_signed_area_distance_{attribution_buffer_m}m_nn_fallback.csv"
        ),
    }


def build_total_summary(
    scenario_name,
    sector_ead_table,
    sector_sign_table,
    discount_factor,
    discount_years,
    discount_rate,
    discounted_suffix,
):
    total_with_mangroves_usd = float(pd.to_numeric(sector_ead_table["EAD_With_Mangroves_USD"], errors="coerce").sum())
    total_without_mangroves_usd = float(
        pd.to_numeric(sector_ead_table["EAD_Without_Mangroves_USD"], errors="coerce").sum()
    )
    total_net_avoided_usd = float(pd.to_numeric(sector_ead_table["Avoided_EAD_USD"], errors="coerce").sum())
    total_positive_avoided_usd = float(
        pd.to_numeric(sector_sign_table["Positive_Avoided_EAD_USD"], errors="coerce").sum()
    )
    total_negative_avoided_usd = float(
        pd.to_numeric(sector_sign_table["Negative_Avoided_EAD_USD"], errors="coerce").sum()
    )
    total_gross_avoided_usd = float(
        pd.to_numeric(sector_sign_table["Gross_Avoided_EAD_USD"], errors="coerce").sum()
    )
    total_positive_attributed_usd = float(
        pd.to_numeric(sector_sign_table["Positive_Attributed_EAD_USD"], errors="coerce").sum()
    )
    total_negative_attributed_usd = float(
        pd.to_numeric(sector_sign_table["Negative_Attributed_EAD_USD"], errors="coerce").sum()
    )
    total_net_attributed_usd = float(
        pd.to_numeric(sector_sign_table["Net_Attributed_EAD_USD"], errors="coerce").sum()
    )
    total_gross_attributed_usd = float(
        pd.to_numeric(sector_sign_table["Gross_Attributed_EAD_USD"], errors="coerce").sum()
    )
    total_increased_damage_usd = abs(total_negative_attributed_usd)

    total_summary = pd.DataFrame(
        [
            {
                "Scenario": scenario_name,
                "Discount_Years": discount_years,
                "Discount_Rate": discount_rate,
                "Discount_Factor": discount_factor,
                "EAD_With_Mangroves_USD": total_with_mangroves_usd,
                "EAD_Without_Mangroves_USD": total_without_mangroves_usd,
                "Net_Avoided_EAD_USD": total_net_avoided_usd,
                "Positive_Avoided_EAD_USD": total_positive_avoided_usd,
                "Negative_Avoided_EAD_USD": total_negative_avoided_usd,
                "Gross_Avoided_EAD_USD": total_gross_avoided_usd,
                "Positive_Attributed_EAD_USD": total_positive_attributed_usd,
                "Negative_Attributed_EAD_USD": total_negative_attributed_usd,
                "Net_Attributed_EAD_USD": total_net_attributed_usd,
                "Gross_Attributed_EAD_USD": total_gross_attributed_usd,
                "Increased_Damage_USD": total_increased_damage_usd,
                "Percent_Net_Avoided_vs_NoMangroves": (
                    100.0 * total_net_avoided_usd / total_without_mangroves_usd
                    if total_without_mangroves_usd
                    else np.nan
                ),
                "Percent_Gross_Positive_Avoided_vs_NoMangroves": (
                    100.0 * total_positive_avoided_usd / total_without_mangroves_usd
                    if total_without_mangroves_usd
                    else np.nan
                ),
                "Percent_Increased_Damage_vs_NoMangroves": (
                    100.0 * total_increased_damage_usd / total_without_mangroves_usd
                    if total_without_mangroves_usd
                    else np.nan
                ),
            }
        ]
    )

    return add_discounted_columns(
        total_summary,
        [
            "EAD_With_Mangroves_USD",
            "EAD_Without_Mangroves_USD",
            "Net_Avoided_EAD_USD",
            "Positive_Avoided_EAD_USD",
            "Negative_Avoided_EAD_USD",
            "Gross_Avoided_EAD_USD",
            "Positive_Attributed_EAD_USD",
            "Negative_Attributed_EAD_USD",
            "Net_Attributed_EAD_USD",
            "Gross_Attributed_EAD_USD",
            "Increased_Damage_USD",
        ],
        discount_factor,
        discounted_suffix,
    )


def build_sector_summary_table(scenario_name, discounted_tables, discounted_suffix):
    sector_ead_table = discounted_tables["sector_ead"][
        [
            "Sector",
            "EAD_With_Mangroves_USD",
            "EAD_Without_Mangroves_USD",
            "Avoided_EAD_USD",
            f"EAD_With_Mangroves_USD_{discounted_suffix}",
            f"EAD_Without_Mangroves_USD_{discounted_suffix}",
            f"Avoided_EAD_USD_{discounted_suffix}",
            "Percent_Avoided_EAD_vs_NoMangroves",
        ]
    ]
    sector_sign_table = discounted_tables["sector_sign"][
        [
            "Sector",
            "Positive_Attributed_EAD_USD",
            "Negative_Attributed_EAD_USD",
            "Net_Attributed_EAD_USD",
            "Negative_Attributed_EAD_USD_abs",
            "Gross_Attributed_EAD_USD",
            f"Positive_Attributed_EAD_USD_{discounted_suffix}",
            f"Negative_Attributed_EAD_USD_{discounted_suffix}",
            f"Net_Attributed_EAD_USD_{discounted_suffix}",
            f"Negative_Attributed_EAD_USD_abs_{discounted_suffix}",
            f"Gross_Attributed_EAD_USD_{discounted_suffix}",
            "Net_Impact_Class",
        ]
    ]
    sector_summary_table = sector_ead_table.merge(sector_sign_table, on="Sector", how="left")
    sector_summary_table.insert(0, "Scenario", scenario_name)
    sector_summary_table["Increased_Damage_USD"] = sector_summary_table["Negative_Attributed_EAD_USD_abs"]
    sector_summary_table[f"Increased_Damage_USD_{discounted_suffix}"] = sector_summary_table[
        f"Negative_Attributed_EAD_USD_abs_{discounted_suffix}"
    ]
    sector_summary_table["Percent_Increased_Damage_vs_NoMangroves"] = np.where(
        sector_summary_table["EAD_Without_Mangroves_USD"] > 0,
        100.0
        * sector_summary_table["Increased_Damage_USD"]
        / sector_summary_table["EAD_Without_Mangroves_USD"],
        np.nan,
    )
    return sector_summary_table


def build_subsector_summary_table(scenario_name, discounted_tables, discounted_suffix):
    subsector_ead_table = discounted_tables["subsector_ead"][
        [
            "Sector",
            "Subsector",
            "EAD_With_Mangroves_USD",
            "EAD_Without_Mangroves_USD",
            "Avoided_EAD_USD",
            f"EAD_With_Mangroves_USD_{discounted_suffix}",
            f"EAD_Without_Mangroves_USD_{discounted_suffix}",
            f"Avoided_EAD_USD_{discounted_suffix}",
            "Percent_Avoided_EAD_vs_NoMangroves",
        ]
    ]
    subsector_sign_table = discounted_tables["subsector_sign"][
        [
            "Sector",
            "Subsector",
            "Positive_Attributed_EAD_USD",
            "Negative_Attributed_EAD_USD",
            "Net_Attributed_EAD_USD",
            "Negative_Attributed_EAD_USD_abs",
            "Gross_Attributed_EAD_USD",
            f"Positive_Attributed_EAD_USD_{discounted_suffix}",
            f"Negative_Attributed_EAD_USD_{discounted_suffix}",
            f"Net_Attributed_EAD_USD_{discounted_suffix}",
            f"Negative_Attributed_EAD_USD_abs_{discounted_suffix}",
            f"Gross_Attributed_EAD_USD_{discounted_suffix}",
            "Net_Impact_Class",
        ]
    ]
    subsector_summary_table = subsector_ead_table.merge(
        subsector_sign_table,
        on=["Sector", "Subsector"],
        how="left",
    )
    subsector_summary_table.insert(0, "Scenario", scenario_name)
    subsector_summary_table["Increased_Damage_USD"] = subsector_summary_table[
        "Negative_Attributed_EAD_USD_abs"
    ]
    subsector_summary_table[f"Increased_Damage_USD_{discounted_suffix}"] = subsector_summary_table[
        f"Negative_Attributed_EAD_USD_abs_{discounted_suffix}"
    ]
    subsector_summary_table["Percent_Increased_Damage_vs_NoMangroves"] = np.where(
        subsector_summary_table["EAD_Without_Mangroves_USD"] > 0,
        100.0
        * subsector_summary_table["Increased_Damage_USD"]
        / subsector_summary_table["EAD_Without_Mangroves_USD"],
        np.nan,
    )
    return subsector_summary_table


def build_patch_summary_table(scenario_name, discounted_tables):
    patch_summary_table = discounted_tables["mangrove_patch"].copy()
    patch_summary_table.insert(0, "Scenario", scenario_name)
    return patch_summary_table

In [ ]:
monetary_columns_by_table = {
    "sector_ead": [
        "EAD_With_Mangroves_USD",
        "EAD_Without_Mangroves_USD",
        "Avoided_EAD_USD",
    ],
    "subsector_ead": [
        "EAD_With_Mangroves_USD",
        "EAD_Without_Mangroves_USD",
        "Avoided_EAD_USD",
    ],
    "asset_ead": [
        "EAD_With_Mangroves_USD",
        "EAD_Without_Mangroves_USD",
        "Avoided_EAD_USD",
    ],
    "sector_attribution": [
        "Avoided_EAD_USD",
        "Attributed_EAD_USD",
        "Unattributed_EAD_USD",
    ],
    "subsector_attribution": [
        "Avoided_EAD_USD",
        "Attributed_EAD_USD",
        "Unattributed_EAD_USD",
    ],
    "sector_sign": [
        "Positive_Avoided_EAD_USD",
        "Negative_Avoided_EAD_USD",
        "Net_Avoided_EAD_USD",
        "Positive_Attributed_EAD_USD",
        "Negative_Attributed_EAD_USD",
        "Net_Attributed_EAD_USD",
        "Absolute_Attributed_EAD_USD",
        "Negative_Avoided_EAD_USD_abs",
        "Negative_Attributed_EAD_USD_abs",
        "Gross_Avoided_EAD_USD",
        "Gross_Attributed_EAD_USD",
    ],
    "subsector_sign": [
        "Positive_Avoided_EAD_USD",
        "Negative_Avoided_EAD_USD",
        "Net_Avoided_EAD_USD",
        "Positive_Attributed_EAD_USD",
        "Negative_Attributed_EAD_USD",
        "Net_Attributed_EAD_USD",
        "Absolute_Attributed_EAD_USD",
        "Negative_Avoided_EAD_USD_abs",
        "Negative_Attributed_EAD_USD_abs",
        "Gross_Avoided_EAD_USD",
        "Gross_Attributed_EAD_USD",
    ],
    "mangrove_patch": [
        "Net_Avoided_EAD_USD_attributed",
        "Positive_Avoided_EAD_USD_attributed",
        "Negative_Avoided_EAD_USD_attributed",
        "Absolute_Avoided_EAD_USD_attributed",
        "Negative_Avoided_EAD_USD_attributed_abs",
        "Gross_Avoided_EAD_USD_attributed",
    ],
    "asset_attribution_status": [
        "Avoided_EAD_USD",
        "Attributed_EAD_USD",
        "Unattributed_EAD_USD",
    ],
}

output_file_names_by_table = {
    "sector_ead": "coastal_ead_sector_summary_usd_with_pct_avoided_50yr_discounted_10pct.csv",
    "subsector_ead": "coastal_ead_subsector_summary_usd_with_pct_avoided_50yr_discounted_10pct.csv",
    "asset_ead": "coastal_ead_asset_level_usd_50yr_discounted_10pct.csv",
    "sector_attribution": "attribution_breakdown_by_sector_signed_area_distance_5000m_nn_fallback_50yr_discounted_10pct.csv",
    "subsector_attribution": "attribution_breakdown_by_subsector_signed_area_distance_5000m_nn_fallback_50yr_discounted_10pct.csv",
    "sector_sign": "sector_attribution_net_sign_profile_signed_area_distance_5000m_nn_fallback_50yr_discounted_10pct.csv",
    "subsector_sign": "subsector_attribution_net_sign_profile_signed_area_distance_5000m_nn_fallback_50yr_discounted_10pct.csv",
    "mangrove_patch": "mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback_50yr_discounted_10pct.csv",
    "asset_attribution_status": "asset_attribution_status_signed_area_distance_5000m_nn_fallback_50yr_discounted_10pct.csv",
    "total_summary": "coastal_ead_total_summary_annual_and_50yr_discounted_10pct.csv",
}

total_summary_tables = []
sector_summary_tables = []
subsector_summary_tables = []
patch_summary_tables = []

for scenario_name, scenario_path in scenario_paths.items():
    scenario_tables = load_scenario_tables(scenario_path, mangrove_attribution_buffer_m)
    discounted_tables = {}

    for table_name, scenario_table in scenario_tables.items():
        discounted_tables[table_name] = add_discounted_columns(
            scenario_table,
            monetary_columns_by_table[table_name],
            discount_factor_50_years,
            discounted_suffix,
        )

    discounted_tables["total_summary"] = build_total_summary(
        scenario_name,
        discounted_tables["sector_ead"],
        discounted_tables["sector_sign"],
        discount_factor_50_years,
        discount_years,
        discount_rate,
        discounted_suffix,
    )

    total_summary_tables.append(discounted_tables["total_summary"])
    sector_summary_tables.append(
        build_sector_summary_table(scenario_name, discounted_tables, discounted_suffix)
    )
    subsector_summary_tables.append(
        build_subsector_summary_table(scenario_name, discounted_tables, discounted_suffix)
    )
    patch_summary_tables.append(build_patch_summary_table(scenario_name, discounted_tables))

    for table_name, output_file_name in output_file_names_by_table.items():
        discounted_tables[table_name].to_csv(
            scenario_output_paths[scenario_name] / output_file_name,
            index=False,
        )

total_summary_compare = pd.concat(total_summary_tables, ignore_index=True)
sector_summary_compare = pd.concat(sector_summary_tables, ignore_index=True)
subsector_summary_compare = pd.concat(subsector_summary_tables, ignore_index=True)
patch_summary_compare = pd.concat(patch_summary_tables, ignore_index=True)

total_summary_compare.to_csv(
    comparison_output_dir / "coastal_ead_total_summary_annual_and_50yr_discounted_10pct.csv",
    index=False,
)
sector_summary_compare.to_csv(
    comparison_output_dir / "coastal_ead_sector_summary_min_max_50yr_discounted_10pct.csv",
    index=False,
)
subsector_summary_compare.to_csv(
    comparison_output_dir / "coastal_ead_subsector_summary_min_max_50yr_discounted_10pct.csv",
    index=False,
)
patch_summary_compare.to_csv(
    comparison_output_dir / "mangrove_patch_summary_min_max_50yr_discounted_10pct.csv",
    index=False,
)

pd.DataFrame(
    {
        "Scenario": list(scenario_output_paths.keys()) + ["comparison"],
        "Output_Directory": [str(path) for path in scenario_output_paths.values()] + [str(comparison_output_dir)],
    }
)

## Total 50-year discounted avoided EAD summary across all sectors

In [ ]:
total_summary_display = total_summary_compare.copy()

annual_label_columns = [
    "EAD_Without_Mangroves_USD",
    "EAD_With_Mangroves_USD",
    "Net_Avoided_EAD_USD",
    "Positive_Attributed_EAD_USD",
    "Increased_Damage_USD",
]
for annual_label_column in annual_label_columns:
    total_summary_display[f"{annual_label_column}_Label"] = total_summary_display[
        annual_label_column
    ].apply(format_usd_readable)
    total_summary_display[f"{annual_label_column}_{discounted_suffix}_Label"] = total_summary_display[
        f"{annual_label_column}_{discounted_suffix}"
    ].apply(format_usd_readable)

total_summary_display[
    [
        "Scenario",
        "Discount_Factor",
        "EAD_Without_Mangroves_USD_Label",
        f"EAD_Without_Mangroves_USD_{discounted_suffix}_Label",
        "Net_Avoided_EAD_USD_Label",
        f"Net_Avoided_EAD_USD_{discounted_suffix}_Label",
        "Positive_Attributed_EAD_USD_Label",
        f"Positive_Attributed_EAD_USD_{discounted_suffix}_Label",
        "Increased_Damage_USD_Label",
        f"Increased_Damage_USD_{discounted_suffix}_Label",
        "Percent_Net_Avoided_vs_NoMangroves",
        "Percent_Increased_Damage_vs_NoMangroves",
    ]
].rename(
    columns={
        "EAD_Without_Mangroves_USD_Label": "Annual_NoMangroves",
        f"EAD_Without_Mangroves_USD_{discounted_suffix}_Label": "NoMangroves_PV_50Y_10pct",
        "Net_Avoided_EAD_USD_Label": "Annual_Net_Avoided",
        f"Net_Avoided_EAD_USD_{discounted_suffix}_Label": "Net_Avoided_PV_50Y_10pct",
        "Positive_Attributed_EAD_USD_Label": "Annual_Positive_Attributed",
        f"Positive_Attributed_EAD_USD_{discounted_suffix}_Label": "Positive_Attributed_PV_50Y_10pct",
        "Increased_Damage_USD_Label": "Annual_Increased_Damage",
        f"Increased_Damage_USD_{discounted_suffix}_Label": "Increased_Damage_PV_50Y_10pct",
        "Percent_Net_Avoided_vs_NoMangroves": "Percent_Net_Avoided_vs_NoMangroves",
        "Percent_Increased_Damage_vs_NoMangroves": "Percent_Increased_Damage_vs_NoMangroves",
    }
)

## Sector-level 50-year discounted summary

In [ ]:
sector_summary_display = sector_summary_compare.copy()
sector_summary_display["Avoided_EAD_PV_50Y_10pct_Label"] = sector_summary_display[
    f"Avoided_EAD_USD_{discounted_suffix}"
].apply(format_usd_readable)
sector_summary_display["Positive_Attributed_PV_50Y_10pct_Label"] = sector_summary_display[
    f"Positive_Attributed_EAD_USD_{discounted_suffix}"
].apply(format_usd_readable)
sector_summary_display["Increased_Damage_PV_50Y_10pct_Label"] = sector_summary_display[
    f"Increased_Damage_USD_{discounted_suffix}"
].apply(format_usd_readable)

sector_summary_display = sector_summary_display.sort_values(
    ["Scenario", f"Avoided_EAD_USD_{discounted_suffix}"],
    ascending=[True, False],
).reset_index(drop=True)

sector_summary_display[
    [
        "Scenario",
        "Sector",
        "Avoided_EAD_PV_50Y_10pct_Label",
        "Positive_Attributed_PV_50Y_10pct_Label",
        "Increased_Damage_PV_50Y_10pct_Label",
        "Percent_Avoided_EAD_vs_NoMangroves",
        "Percent_Increased_Damage_vs_NoMangroves",
        "Net_Impact_Class",
    ]
]

## Subsector-level 50-year discounted summary

In [ ]:
subsector_summary_display = subsector_summary_compare.copy()
subsector_summary_display["Avoided_EAD_PV_50Y_10pct_Label"] = subsector_summary_display[
    f"Avoided_EAD_USD_{discounted_suffix}"
].apply(format_usd_readable)
subsector_summary_display["Increased_Damage_PV_50Y_10pct_Label"] = subsector_summary_display[
    f"Increased_Damage_USD_{discounted_suffix}"
].apply(format_usd_readable)

subsector_summary_display = subsector_summary_display.sort_values(
    ["Scenario", f"Avoided_EAD_USD_{discounted_suffix}"],
    ascending=[True, False],
).reset_index(drop=True)

subsector_summary_display[
    [
        "Scenario",
        "Sector",
        "Subsector",
        "Avoided_EAD_PV_50Y_10pct_Label",
        "Increased_Damage_PV_50Y_10pct_Label",
        "Percent_Avoided_EAD_vs_NoMangroves",
        "Percent_Increased_Damage_vs_NoMangroves",
        "Net_Impact_Class",
    ]
]

## Mangrove patches with the highest 50-year discounted net attributed EAD

In [ ]:
patch_summary_display = patch_summary_compare.copy()
patch_summary_display["Net_Attributed_PV_50Y_10pct_Label"] = patch_summary_display[
    f"Net_Avoided_EAD_USD_attributed_{discounted_suffix}"
].apply(format_usd_readable)
patch_summary_display["Positive_Attributed_PV_50Y_10pct_Label"] = patch_summary_display[
    f"Positive_Avoided_EAD_USD_attributed_{discounted_suffix}"
].apply(format_usd_readable)
patch_summary_display["Increased_Damage_PV_50Y_10pct_Label"] = patch_summary_display[
    f"Negative_Avoided_EAD_USD_attributed_abs_{discounted_suffix}"
].apply(format_usd_readable)

top_patch_summary_display = (
    patch_summary_display.sort_values(
        ["Scenario", f"Net_Avoided_EAD_USD_attributed_{discounted_suffix}"],
        ascending=[True, False],
    )
    .groupby("Scenario", group_keys=False)
    .head(20)
    .reset_index(drop=True)
)

top_patch_summary_display[
    [
        "Scenario",
        "Mangrove_ID",
        "Net_Attributed_PV_50Y_10pct_Label",
        "Positive_Attributed_PV_50Y_10pct_Label",
        "Increased_Damage_PV_50Y_10pct_Label",
        "Asset_Count",
        "Fallback_Analysis_Unit_Count",
        "Net_Impact_Class",
    ]
]

## Mangrove patches with the highest 50-year discounted increased damages

In [ ]:
top_negative_patch_summary_display = (
    patch_summary_display.sort_values(
        ["Scenario", f"Negative_Avoided_EAD_USD_attributed_abs_{discounted_suffix}"],
        ascending=[True, False],
    )
    .groupby("Scenario", group_keys=False)
    .head(20)
    .reset_index(drop=True)
)

top_negative_patch_summary_display[
    [
        "Scenario",
        "Mangrove_ID",
        "Increased_Damage_PV_50Y_10pct_Label",
        "Net_Attributed_PV_50Y_10pct_Label",
        "Positive_Attributed_PV_50Y_10pct_Label",
        "Asset_Count",
        "Fallback_Analysis_Unit_Count",
        "Net_Impact_Class",
    ]
]

## Exported outputs

In [ ]:
exported_outputs = pd.DataFrame(
    [
        {
            "Scenario": scenario_name,
            "Output_Directory": str(scenario_output_path),
        }
        for scenario_name, scenario_output_path in scenario_output_paths.items()
    ]
    + [{"Scenario": "comparison", "Output_Directory": str(comparison_output_dir)}]
)

exported_outputs